In [1]:
from aana.api.editor import (
    deployment_output_to_outputs,
    get_all_deployments,
    get_class_names_from_file,
    get_deployment_methods,
    get_function_details,
    get_pydantic_models,
)
from aana.configs.deployments import deployments as all_deployments
from importlib import resources
import os

deployment_class_to_name = {
    deployment.name: deployment_name
    for deployment_name, deployment in all_deployments.items()
}

pydantic_models = get_pydantic_models()
pydantic_model_names = [m.name for _, m in pydantic_models]
pydantic_model_paths = {m.name: f"{path}.{m.name}" for path, m in pydantic_models}

# outputs = {}
# output_dicts = get_deployments_typed_dicts()
# for output in output_dicts:
#     # print(output.name)
#     # print(deployment_output_to_outputs(output))
#     outputs[output.name] = deployment_output_to_outputs(output)
# walk all the files in the aana and get all the typed dicts
typed_dicts = []
for root, dirs, files in os.walk(resources.path("aana", "")):
    for file in files:
        # Check if the file ends with .py
        if file.endswith(".py"):
            path = os.path.join(root, file)
            # print(path)
            classes = get_class_names_from_file(path)
            # keep only TypedDict
            typed_dicts.extend(
                [
                    c
                    for _, c in classes
                    if "TypedDict" in [c.id for c in c.bases if hasattr(c, "id")]
                ]
            )
outputs = {}
for output in typed_dicts:
    # print(output.name)
    # print(deployment_output_to_outputs(output))
    outputs[output.name] = deployment_output_to_outputs(output)

deployments = {}
for deployment in get_all_deployments():
    deployments[deployment_class_to_name[deployment.name]] = {}
    for method in get_deployment_methods(deployment):
        deployments[deployment_class_to_name[deployment.name]][method["name"]] = {
            "inputs": method["args"],
            "is_generator": method["is_generator"],
            # "return_type": method["return_type"],
            "outputs": outputs.get(method["return_type"], []),
        }

available_functions = [
    "aana.utils.video.download_video",
    "aana.utils.video.extract_frames_decord",
    "aana.utils.video.generate_frames_decord",
]

functions = {}
for func in available_functions:
    functions[func] = get_function_details(func, outputs)

/root/.cache/pypoetry/virtualenvs/aana-vIr3-B0u-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-03-11 17:14:38,089	INFO util.py:159 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2024-03-11 17:14:39,636	INFO util.py:159 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
# graph = {
#     "last_node_id": 8,
#     "last_link_id": 7,
#     "nodes": [
#         {
#             "id": 1,
#             "type": "pipeline/input",
#             "pos": [100, 130],
#             "size": {"0": 410, "1": 106},
#             "flags": {},
#             "order": 0,
#             "mode": 0,
#             "inputs": [],
#             "outputs": [
#                 {"links": [1], "name": "video", "slot_index": 0, "type": "string"}
#             ],
#             "title": "video",
#             "properties": {"type": "", "output_name": "output", "is_partial": False},
#             "widgets_values": ["video", "VideoInput", ""],
#         },
#         {
#             "id": 2,
#             "type": "pipeline/input",
#             "pos": [100, 366],
#             "size": {"0": 410, "1": 106},
#             "flags": {},
#             "order": 1,
#             "mode": 0,
#             "inputs": [],
#             "outputs": [
#                 {
#                     "links": [3],
#                     "name": "video_params",
#                     "slot_index": 0,
#                     "type": "string",
#                 }
#             ],
#             "title": "video_params",
#             "properties": {"type": "", "output_name": "output", "is_partial": False},
#             "widgets_values": ["video_params", "VideoParams", ""],
#         },
#         {
#             "id": 3,
#             "type": "pipeline/function",
#             "pos": [771.2000122070312, 130],
#             "size": {"0": 596.4000244140625, "1": 130},
#             "flags": {},
#             "order": 3,
#             "mode": 0,
#             "inputs": [
#                 {
#                     "link": 1,
#                     "name": "video_input [VideoInput | Video]",
#                     "type": "string",
#                 }
#             ],
#             "outputs": [
#                 {
#                     "links": [2],
#                     "name": "output [Video]",
#                     "slot_index": 0,
#                     "type": "string",
#                 }
#             ],
#             "title": "download_video",
#             "properties": {
#                 "name": "download_video",
#                 "function_name": "aana.utils.video.download_video",
#                 "is_partial": False,
#                 "is_generator": False,
#                 "dict_output": False,
#                 "batched": False,
#                 "is_ray_task": True,
#                 "path[output]": "video.video",
#             },
#             "widgets_values": [
#                 "aana.utils.video.download_video",
#                 False,
#                 False,
#                 "video.video",
#             ],
#             "color": "#333333",
#         },
#         {
#             "id": 6,
#             "type": "pipeline/fixed_input",
#             "pos": [100, 602],
#             "size": {"0": 150, "1": 100},
#             "flags": {},
#             "order": 2,
#             "mode": 0,
#             "inputs": [],
#             "outputs": [{"name": "value", "type": "string", "links": None}],
#             "properties": {"value": 4},
#             "widgets_values": [4],
#         },
#         {
#             "id": 8,
#             "type": "pipeline/endpoint",
#             "pos": [100, 1092],
#             "size": {"0": 410, "1": 290},
#             "flags": {},
#             "order": 6,
#             "mode": 0,
#             "inputs": [
#                 {"link": 6, "name": "captions", "type": "string"},
#                 {"link": 7, "name": "timestamps", "type": "string"},
#                 {"link": None, "name": "caption_ids", "type": "string"},
#             ],
#             "outputs": [],
#             "title": "blip2_video_generate",
#             "properties": {
#                 "path": "/video/generate_captions",
#                 "summary": "Generate captions for videos using BLIP2 OPT-2.7B",
#                 "count_outputs": 3,
#                 "output_0": "captions",
#                 "is_streaming_0": False,
#                 "is_streaming_1": False,
#                 "is_streaming_2": False,
#                 "output_1": "timestamps",
#                 "output_2": "caption_ids",
#             },
#         },
#         {
#             "id": 4,
#             "type": "pipeline/function",
#             "pos": [1467.6000366210938, 130],
#             "size": {"0": 579.5999755859375, "1": 286},
#             "flags": {},
#             "order": 4,
#             "mode": 0,
#             "inputs": [
#                 {"link": 2, "name": "video [Video]", "type": "string"},
#                 {"link": 3, "name": "params [VideoParams]", "type": "string"},
#                 {"link": 5, "name": "batch_size [int] = 8", "type": "string"},
#             ],
#             "outputs": [
#                 {
#                     "links": [4],
#                     "name": "frames [list[Image]]",
#                     "slot_index": 0,
#                     "type": "string",
#                 },
#                 {
#                     "links": [],
#                     "name": "frame_ids [list[int]]",
#                     "slot_index": 1,
#                     "type": "string",
#                 },
#                 {
#                     "links": [7],
#                     "name": "timestamps [list[float]]",
#                     "slot_index": 2,
#                     "type": "string",
#                 },
#                 {
#                     "links": [],
#                     "name": "duration [float]",
#                     "slot_index": 3,
#                     "type": "string",
#                 },
#             ],
#             "title": "generate_frames_for_video",
#             "properties": {
#                 "name": "generate_frames_for_video",
#                 "function_name": "aana.utils.video.generate_frames_decord",
#                 "is_partial": False,
#                 "is_generator": True,
#                 "dict_output": True,
#                 "batched": False,
#                 "generator_path": "video.video",
#                 "is_ray_task": True,
#                 "path[duration]": "video.duration",
#                 "path[frame_ids]": "video.frames.[*].id",
#                 "path[frames]": "video.frames.[*].image",
#                 "path[timestamps]": "video.timestamps",
#             },
#             "widgets_values": [
#                 "aana.utils.video.generate_frames_decord",
#                 False,
#                 False,
#                 "video.video",
#                 "video.duration",
#                 "video.frames.[*].id",
#                 "video.frames.[*].image",
#                 "video.timestamps",
#             ],
#             "color": "#6666FF",
#         },
#         {
#             "id": 5,
#             "type": "pipeline/deployment",
#             "pos": [100, 832],
#             "size": {"0": 571.2000122070312, "1": 130},
#             "flags": {},
#             "order": 5,
#             "mode": 0,
#             "inputs": [{"link": 4, "name": "images [list[Image]]", "type": "string"}],
#             "outputs": [
#                 {
#                     "links": [6],
#                     "name": "captions [CaptionsList]",
#                     "slot_index": 0,
#                     "type": "string",
#                 }
#             ],
#             "title": "hf_blip2_opt_2_7b_video",
#             "properties": {
#                 "name": "hf_blip2_opt_2_7b_video",
#                 "deployment_name": "hf_blip2_deployment_opt_2_7b",
#                 "method": "generate_batch",
#                 "is_partial": True,
#                 "is_generator": False,
#                 "flatten_by": "",
#                 "path[captions]": "video.frames.[*].caption_hf_blip2_opt_2_7b",
#             },
#             "widgets_values": [
#                 "hf_blip2_opt_2_7b_video",
#                 "generate_batch",
#                 True,
#                 "video.frames.[*].caption_hf_blip2_opt_2_7b",
#             ],
#             "color": "#FF6666",
#         },
#     ],
#     "links": [
#         [1, 1, 0, 3, 0, "string"],
#         [2, 3, 0, 4, 0, "string"],
#         [3, 2, 0, 4, 1, "string"],
#         [4, 4, 0, 5, 0, "string"],
#         [5, 6, 0, 4, 2, "string"],
#         [6, 5, 0, 8, 0, "string"],
#         [7, 4, 2, 8, 1, "string"],
#     ],
#     "groups": [],
#     "config": {},
#     "extra": {},
#     "version": 0.4,
# }

In [3]:
# graph = {
#     "last_node_id": 6,
#     "last_link_id": 4,
#     "nodes": [
#         {
#             "id": 1,
#             "type": "pipeline/input",
#             "pos": [100, 130],
#             "size": {"0": 410, "1": 106},
#             "flags": {},
#             "order": 0,
#             "mode": 0,
#             "outputs": [
#                 {
#                     "name": "video [VideoInput]",
#                     "type": "string",
#                     "links": [1],
#                     "slot_index": 0,
#                 }
#             ],
#             "title": "Input: video",
#             "properties": {"type": "VideoInput", "output_name": "video"},
#             "widgets_values": ["video", "VideoInput", ""],
#         },
#         {
#             "id": 3,
#             "type": "pipeline/function",
#             "pos": [610, 130],
#             "size": {"0": 596.4000244140625, "1": 130},
#             "flags": {},
#             "order": 2,
#             "mode": 0,
#             "inputs": [
#                 {
#                     "name": "video_input [VideoInput | Video]",
#                     "type": "string",
#                     "link": 1,
#                 }
#             ],
#             "outputs": [
#                 {
#                     "name": "output [Video]",
#                     "type": "string",
#                     "links": [2],
#                     "slot_index": 0,
#                 }
#             ],
#             "title": "aana.utils.video.download_video",
#             "properties": {
#                 "name": "function",
#                 "function_name": "aana.utils.video.download_video",
#                 "is_partial": False,
#                 "is_generator": False,
#                 "dict_output": False,
#                 "batched": False,
#                 "path[output]": "",
#             },
#             "widgets_values": ["aana.utils.video.download_video", False, False, ""],
#             "color": "#333333",
#         },
#         {
#             "id": 5,
#             "type": "pipeline/input",
#             "pos": [100, 366],
#             "size": {"0": 410, "1": 106},
#             "flags": {},
#             "order": 1,
#             "mode": 0,
#             "outputs": [
#                 {
#                     "name": "whisper_params [WhisperParams]",
#                     "type": "string",
#                     "links": [3],
#                     "slot_index": 0,
#                 }
#             ],
#             "title": "Input: whisper_params",
#             "properties": {"type": "WhisperParams", "output_name": "whisper_params"},
#             "widgets_values": ["whisper_params", "WhisperParams", ""],
#         },
#         {
#             "id": 2,
#             "type": "pipeline/deployment",
#             "pos": [1306.4000244140625, 130],
#             "size": {"0": 798, "1": 242},
#             "flags": {},
#             "order": 3,
#             "mode": 0,
#             "inputs": [
#                 {"name": "media [Video]", "type": "string", "link": 2},
#                 {"name": "params [WhisperParams | None]", "type": "string", "link": 3},
#             ],
#             "outputs": [
#                 {
#                     "name": "segments [list[AsrSegment]]",
#                     "type": "string",
#                     "links": [4],
#                     "slot_index": 0,
#                 },
#                 {
#                     "name": "transcription_info [AsrTranscriptionInfo]",
#                     "type": "string",
#                     "links": None,
#                 },
#                 {
#                     "name": "transcription [AsrTranscription]",
#                     "type": "string",
#                     "links": None,
#                 },
#             ],
#             "title": "whisper_deployment_medium_transcribe_stream",
#             "properties": {
#                 "name": "deployment",
#                 "deployment_name": "whisper_deployment_medium",
#                 "method": "transcribe_stream",
#                 "is_partial": False,
#                 "is_generator": True,
#                 "flatten_by": "",
#                 "generator_path": "",
#                 "path[segments]": "",
#                 "path[transcription_info]": "",
#                 "path[transcription]": "",
#             },
#             "widgets_values": [
#                 "whisper_deployment_medium",
#                 "transcribe_stream",
#                 False,
#                 "",
#                 "",
#                 "",
#                 "",
#             ],
#             "color": "#6666FF",
#         },
#         {
#             "id": 6,
#             "type": "pipeline/endpoint",
#             "pos": [2204.4000244140625, 130],
#             "size": {"0": 410, "1": 154},
#             "flags": {},
#             "order": 4,
#             "mode": 0,
#             "inputs": [{"name": "segments", "type": "string", "link": 4}],
#             "title": "Endpoint: /transcribe",
#             "properties": {
#                 "path": "",
#                 "summary": "",
#                 "count_outputs": 1,
#                 "output_0": "segments",
#                 "is_streaming_0": True,
#             },
#         },
#     ],
#     "links": [
#         [1, 1, 0, 3, 0, "string"],
#         [2, 3, 0, 2, 0, "string"],
#         [3, 5, 0, 2, 1, "string"],
#         [4, 2, 0, 6, 0, "string"],
#     ],
#     "groups": [],
#     "config": {},
#     "extra": {},
#     "version": 0.4,
# }

In [16]:
graph = {
    "last_node_id": 1,
    "last_link_id": 0,
    "nodes": [
        {
            "id": 1,
            "type": "pipeline/input",
            "pos": [125, 119],
            "size": {"0": 410, "1": 106},
            "flags": {},
            "order": 0,
            "mode": 0,
            "outputs": [
                {"name": "output [VideoInput]", "type": "string", "links": None}
            ],
            "title": "Input: output",
            "properties": {"type": "VideoInput", "output_name": "output"},
            "widgets_values": ["output", "VideoInput", ""],
        }
    ],
    "links": [],
    "groups": [],
    "config": {},
    "extra": {},
    "version": 0.4,
}

In [17]:
node_id_to_node = {node["id"]: node for node in graph["nodes"]}

In [18]:
from collections import defaultdict
from typing import Any


all_outputs = {}
output_by_location = defaultdict(dict)
for node in graph["nodes"]:
    if node["type"] == "pipeline/fixed_input":
        pass

    elif node["type"] == "pipeline/input":
        output_name, output_type, path = node["widgets_values"]

        for output in node["outputs"]:
            name = output["name"]

            if path == "":
                path = output_name

            # output_name = node["title"].replace(" ", "_").lower() + "_" + name
            k = 2
            while output_name in all_outputs:
                output_name = (
                    node["title"].replace(" ", "_").lower() + "_" + name + f"_{k}"
                )

            output_def = {
                "name": output_name,
                "key": output_name,
                "path": path,
                "data_model": pydantic_model_paths[output_type],
            }

            all_outputs[output_name] = output_def

            if "slot_index" in output:
                output_by_location[node["id"]][output["slot_index"]] = output_def
    else:
        for output in node.get("outputs", []):
            name_with_type = output["name"]
            name = name_with_type.split("[")[0].strip()

            output_name = node["title"].replace(" ", "_").lower() + "_" + name
            k = 2
            while output_name in all_outputs:
                output_name = (
                    node["title"].replace(" ", "_").lower() + "_" + name + f"_{k}"
                )

            output_type = name_with_type.split("[")[1].split("]")[0]
            path = node["properties"][f"path[{name}]"]
            if path == "":
                path = output_name

            output_def = {
                "name": output_name,
                "key": name,
                "path": path,
                # "data_model": pydantic_model_paths.get(output_type, Any),
            }
            if output_type in pydantic_model_paths:
                output_def["data_model"] = pydantic_model_paths[output_type]

            all_outputs[output_name] = output_def

            if "slot_index" in output:
                output_by_location[node["id"]][output["slot_index"]] = output_def

In [19]:
node

{'id': 1,
 'type': 'pipeline/input',
 'pos': [125, 119],
 'size': {'0': 410, '1': 106},
 'flags': {},
 'order': 0,
 'mode': 0,
 'outputs': [{'name': 'output [VideoInput]', 'type': 'string', 'links': None}],
 'title': 'Input: output',
 'properties': {'type': 'VideoInput', 'output_name': 'output'},
 'widgets_values': ['output', 'VideoInput', '']}

In [20]:
all_outputs

{'input:_output_output [VideoInput]': {'name': 'input:_output_output [VideoInput]',
  'key': 'input:_output_output [VideoInput]',
  'path': 'output',
  'data_model': 'aana.models.pydantic.video_input.VideoInput'}}

In [21]:
links = {}
for link in graph["links"]:
    link_id, origin_node_id, origin_slot, target_node_id, target_slot, _ = link
    links[link_id] = {
        "origin_node": origin_node_id,
        "origin_slot": origin_slot,
        "target_node": target_node_id,
        "target_slot": target_slot,
    }

In [22]:
kwargs = defaultdict(dict)
for link_id, link in links.items():
    origin_node = node_id_to_node[link["origin_node"]]
    if origin_node["type"] == "pipeline/fixed_input":
        origin_output = node_id_to_node[link["origin_node"]]["outputs"][
            link["origin_slot"]
        ]
        target_input = node_id_to_node[link["target_node"]]["inputs"][
            link["target_slot"]
        ]

        kwargs_name = target_input["name"].split("[")[0].strip()

        kwargs[link["target_node"]][kwargs_name] = origin_node["properties"]["value"]

In [23]:
kwargs

defaultdict(dict, {})

In [24]:
from typing import Any


pipeline = []
endpoints = []

for node in graph["nodes"]:
    if node["type"] == "pipeline/fixed_input":
        continue
    elif node["type"] == "pipeline/input":
        pipeline_node = {
            "name": node["title"].replace("Input: ", "input_"),
            "type": "input",
            "inputs": [],
        }
        # break
        pipeline_node["name"] = output_by_location[node["id"]][0]["name"]
        # output_name, output_type, path = node["widgets_values"]

        # outputs = []
        # for output in node["outputs"]:
        #     outputs.append(
        #         {
        #             "name": output["name"],
        #             "key": output["name"],
        #             "path": path if path else output["name"],
        #             "data_model": output_type,
        #         }
        #     )

        # pipeline_node["outputs"] = outputs
    elif node["type"] == "pipeline/deployment":
        pipeline_node = {
            "name": node["title"].replace(" ", "_"),
            "type": "ray_deployment",
            "deployment_name": node["properties"]["deployment_name"],
            "method": node["properties"]["method"],
        }
    elif node["type"] == "pipeline/function":
        pipeline_node = {
            "name": node["title"].replace(" ", "_"),
            "function": node["properties"]["function_name"],
            "type": "ray_task" if node["properties"]["is_ray_task"] else "function",
            "dict_output": node["properties"]["dict_output"],
            "batched": node["properties"]["batched"],
        }
        # outputs = []
        # for output in node["outputs"]:
        #     name_with_type = output["name"]
        #     name = name_with_type.split("[")[0].strip()
        #     output_type = name_with_type.split("[")[1].split("]")[0]
        #     path = node["properties"][f"path[{name}]"]
        #     if path == "":
        #         path = name
        #     outputs.append(
        #         {
        #             "name": name,
        #             "key": name,
        #             "path": path,
        #             "data_model": pydantic_model_paths.get(output_type, Any),
        #         }
        #     )
        # pipeline_node["outputs"] = outputs
    elif node["type"] == "pipeline/endpoint":
        pipeline_node = None
        endpoint = {
            "name": node["title"].replace(" ", "_"),
            "path": node["properties"]["path"],
            "summary": node["properties"]["summary"],
            "streaming": False,
        }
        endpoint_outputs = []
        for input_index, input in enumerate(node["inputs"]):
            if input["link"] is None:
                continue
            link = links[input["link"]]
            output_def = output_by_location[link["origin_node"]][link["origin_slot"]]
            output_name = node["properties"][f"output_{input_index}"]
            is_streaming = node["properties"][f"is_streaming_{input_index}"]
            endpoint_outputs.append(
                {
                    "name": output_name,
                    "output": output_def["name"],
                    "streaming": is_streaming,
                }
            )
            if is_streaming:
                endpoint["streaming"] = True
        endpoint["outputs"] = endpoint_outputs
        endpoints.append(endpoint)
    else:
        # continue
        # break
        raise ValueError(f"Unknown node type: {node['type']}")

    if node["type"] in ["pipeline/function", "pipeline/deployment"]:
        if (
            "flatten_by" in node["properties"]
            and node["properties"]["flatten_by"] != ""
        ):
            pipeline_node["flatten_by"] = node["properties"]["flatten_by"]

        is_partial = node["properties"].get("is_partial", False)

        if node["properties"].get("is_generator", False):
            pipeline_node["data_type"] = "generator"
            pipeline_node["generator_path"] = node["properties"]["generator_path"]

        if node["id"] in kwargs:
            pipeline_node["kwargs"] = kwargs[node["id"]]

        inputs = []
        for input in node["inputs"]:
            link = links[input["link"]]
            if (
                link["origin_node"] not in output_by_location
                or link["origin_slot"] not in output_by_location[link["origin_node"]]
            ):
                continue
            output_def = output_by_location[link["origin_node"]][link["origin_slot"]]
            key = input["name"].split("[")[0].strip()
            input_def = {
                "name": output_def["name"],
                "key": key,
                "path": output_def["path"],
            }
            if is_partial:
                input_def["partial"] = True
            inputs.append(input_def)

        pipeline_node["inputs"] = inputs

    if node["type"] in ["pipeline/function", "pipeline/deployment", "pipeline/input"]:
        outputs = []
        for output in node["outputs"]:
            # output_def = output_by_location[node["id"]][output["slot_index"]]
            # if output["slot_index"] in output_by_location[node["id"]]:
            output_def = output_by_location[node["id"]][output["slot_index"]]
            outputs.append(output_def)

        pipeline_node["outputs"] = outputs

    if pipeline_node:
        pipeline.append(pipeline_node)

KeyError: 0

In [25]:
output_by_location

defaultdict(dict, {1: {}})

In [17]:
pipeline

[{'name': 'video_video',
  'type': 'input',
  'inputs': [],
  'outputs': [{'name': 'video_video',
    'key': 'video_video',
    'path': 'video',
    'data_model': 'aana.models.pydantic.video_input.VideoInput'}]},
 {'name': 'video_params_video_params',
  'type': 'input',
  'inputs': [],
  'outputs': [{'name': 'video_params_video_params',
    'key': 'video_params_video_params',
    'path': 'video_params',
    'data_model': 'aana.models.pydantic.video_params.VideoParams'}]},
 {'name': 'download_video',
  'function': 'aana.utils.video.download_video',
  'type': 'ray_task',
  'dict_output': False,
  'batched': False,
  'inputs': [{'name': 'video_video', 'key': 'video_input', 'path': 'video'}],
  'outputs': [{'name': 'download_video_output',
    'key': 'output',
    'path': 'video.video'}]},
 {'name': 'generate_frames_for_video',
  'function': 'aana.utils.video.generate_frames_decord',
  'type': 'ray_task',
  'dict_output': True,
  'batched': False,
  'data_type': 'generator',
  'generator_p

In [18]:
endpoints

[{'name': 'blip2_video_generate',
  'path': '/video/generate_captions',
  'summary': 'Generate captions for videos using BLIP2 OPT-2.7B',
  'streaming': False,
  'outputs': [{'name': 'captions',
    'output': 'hf_blip2_opt_2_7b_video_captions',
    'streaming': False},
   {'name': 'timestamps',
    'output': 'generate_frames_for_video_timestamps',
    'streaming': False}]}]

In [19]:
pipeline_name = "video_captioning"
pipeline_file = resources.path("aana.configs.pipelines", f"{pipeline_name}.json")
endpoints_file = resources.path("aana.configs.endpoints", f"{pipeline_name}.json")

In [20]:
import json

with pipeline_file.open("w") as f:
    json.dump(pipeline, f, indent=2)

with endpoints_file.open("w") as f:
    json.dump(endpoints, f, indent=2)